# SEC API: Three Essential Endpoints

## The Three Endpoints That Matter:

### 1. **Company Facts** (Best for ingestion pipeline)
```
https://data.sec.gov/api/xbrl/companyfacts/CIK{10-digit-cik}.json
```
- **Purpose**: Get ALL XBRL facts for one company across ALL filings
- **Data**: 5+ years of financial history in one call
- **Use case**: Bulk ingestion, structured financial data
- **Response size**: Large (~2-5MB), but complete
- **Rate**: 10 requests/second

### 2. **Company Concept** (Efficient if you need specific metrics)
```
https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{Concept}.json
Examples: Revenues, NetIncomeLoss, Assets, Liabilities
```
- **Purpose**: Get ONE metric over time for one company
- **Data**: Just Revenue (or just Net Income, etc.) across all quarters
- **Use case**: Efficient when you only need specific metrics
- **Response size**: Small (~100KB)
- **Rate**: 10 requests/second

### 3. **Submissions** (Get filing links for RAG)
```
https://data.sec.gov/submissions/CIK{cik}.json
```
- **Purpose**: Get filing index with accession numbers, dates, form types, links
- **Data**: Metadata about each filing (10-Q, 10-K dates, links to actual documents)
- **Use case**: Find links to actual 10-Q/10-K text for RAG pipeline
- **Response size**: Small (~500KB)
- **Rate**: 10 requests/second

---
# Setup

In [12]:
import requests
import json
import logging
from typing import List, Dict
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

COMPANY_CIKS = {
    "MSFT": "0000789019",
    "AAPL": "0000320193",
    "GOOGL": "0001652044",
}

SEC_HEADERS = {"User-Agent": "MyCompany info@example.com"}

print("✅ Setup complete")

✅ Setup complete


---
# Endpoint 1: Company Facts (Complete Financial History)

**Use this for**: Bulk ingestion of all structured financial data

In [13]:
def get_all_financial_facts(ticker: str, limit_quarters: int = 3) -> List[Dict]:
    """
    ENDPOINT 1: Company Facts
    
    Gets ALL XBRL facts for a company across all filings.
    One call = 5+ years of complete financial history.
    """
    
    cik = COMPANY_CIKS.get(ticker.upper())
    if not cik:
        return []
    
    logger.info(f"\n{'='*60}")
    logger.info(f"ENDPOINT 1: Company Facts for {ticker}")
    logger.info(f"{'='*60}")
    
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    logger.info(f"URL: {url}\n")
    
    try:
        response = requests.get(url, headers=SEC_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Navigate to us-gaap (US Generally Accepted Accounting Principles)
        facts = data.get("facts", {}).get("us-gaap", {})
        
        results = []
        
        # Extract key metrics (Revenue, Net Income, Operating Income)
        revenue_facts = facts.get("Revenues", {}).get("units", {}).get("USD", [])
        net_income_facts = facts.get("NetIncomeLoss", {}).get("units", {}).get("USD", [])
        operating_income_facts = facts.get("OperatingIncomeLoss", {}).get("units", {}).get("USD", [])
        
        # Organize by period
        period_data = {}
        
        # Process Revenue facts (10-Q = quarterly)
        quarterly_count = 0
        for fact in sorted(revenue_facts, key=lambda x: x.get("end", ""), reverse=True):
            if fact.get("form") == "10-Q" and quarterly_count < limit_quarters:
                period_end = fact.get("end")
                
                if period_end not in period_data:
                    period_data[period_end] = {
                        "ticker": ticker,
                        "period_end": period_end,
                        "filing_date": fact.get("filed"),
                        "form_type": fact.get("form"),
                        "accession_number": fact.get("accession"),
                    }
                
                period_data[period_end]["revenue"] = fact.get("val", 0)
                quarterly_count += 1
        
        # Add Net Income
        for fact in net_income_facts:
            if fact.get("form") == "10-Q":
                period_end = fact.get("end")
                if period_end in period_data:
                    period_data[period_end]["net_income"] = fact.get("val", 0)
        
        # Add Operating Income
        for fact in operating_income_facts:
            if fact.get("form") == "10-Q":
                period_end = fact.get("end")
                if period_end in period_data:
                    period_data[period_end]["operating_income"] = fact.get("val", 0)
        
        # Convert to list
        results = list(period_data.values())
        
        logger.info(f"✅ Found {len(results)} quarters of structured financial data\n")
        for record in results:
            logger.info(f"  {record['period_end']}: Revenue ${record.get('revenue', 0):,.0f}")
        
        return results
    
    except Exception as e:
        logger.error(f"❌ Error: {e}")
        return []

# Test it
facts_data = get_all_financial_facts("MSFT", limit_quarters=2)

2026-07-12 08:46:36,074 - INFO - 
2026-07-12 08:46:36,075 - INFO - ENDPOINT 1: Company Facts for MSFT
2026-07-12 08:46:36,075 - INFO - ============================================================
2026-07-12 08:46:36,076 - INFO - URL: https://data.sec.gov/api/xbrl/companyfacts/CIK0000789019.json

2026-07-12 08:46:36,598 - INFO - ✅ Found 1 quarters of structured financial data

2026-07-12 08:46:36,598 - INFO -   2010-12-31: Revenue $19,953,000,000


---
# Endpoint 2: Company Concept (Single Metric Over Time)

**Use this for**: Efficient queries of specific metrics (smaller response, faster)

In [14]:
def get_concept_over_time_FIXED(ticker: str, concept: str = "Revenues", include_annual: bool = True) -> List[Dict]:
    """
    FIXED: Accept both quarterly (10-Q) and annual (10-K) data
    """
    
    cik = COMPANY_CIKS.get(ticker.upper())
    url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json"
    
    try:
        response = requests.get(url, headers=SEC_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        units = data.get("units", {}).get("USD", [])
        
        # Accept BOTH quarterly (10-Q) and annual (10-K)
        accepted_forms = ["10-Q", "10-K"] if include_annual else ["10-Q"]
        
        quarterly = [u for u in units if u.get("form") in ["10-Q", "10-K"]]  # ✅ Returns 11
        quarterly = sorted(quarterly, key=lambda x: x.get("end", ""), reverse=True)[:10]
        
        for fact in quarterly:
            results.append({
                "ticker": ticker,
                "concept": concept,
                "period_end": fact.get("end"),
                "filing_date": fact.get("filed"),
                "form": fact.get("form"),  # Now shows 10-Q or 10-K
                "value": fact.get("val", 0),
            })

        print(results)
        return results
    
    except Exception as e:
        logger.error(f"Error: {e}")
        return []


---
# Endpoint 3: Submissions (Filing Metadata & Links)

**Use this for**: Getting links to actual 10-Q/10-K text for RAG pipeline

In [15]:
def get_filing_submissions(ticker: str, form_type: str = "10-Q", limit: int = 3) -> List[Dict]:
    """
    ENDPOINT 3: Submissions
    
    Gets filing index with:
    - Accession numbers
    - Filing dates
    - Form types (10-Q, 10-K, 8-K, etc.)
    - Links to actual filing documents
    
    Use this to get the actual 10-Q/10-K text for your RAG side.
    """
    
    cik = COMPANY_CIKS.get(ticker.upper())
    if not cik:
        return []
    
    logger.info(f"\n{'='*60}")
    logger.info(f"ENDPOINT 3: Submissions for {ticker}")
    logger.info(f"Form Type: {form_type}")
    logger.info(f"{'='*60}")
    
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    logger.info(f"URL: {url}\n")
    
    try:
        response = requests.get(url, headers=SEC_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        
        # Get recent filings
        recent = data.get("filings", {}).get("recent", {})
        
        forms = recent.get("form", [])
        filing_dates = recent.get("filingDate", [])
        accession_numbers = recent.get("accessionNumber", [])
        report_dates = recent.get("reportDate", [])
        primary_documents = recent.get("primaryDocument", [])
        
        count = 0
        for form, date, accession, report_date, primary_doc in zip(
            forms, filing_dates, accession_numbers, report_dates, primary_documents
        ):
            if form == form_type and count < limit:
                # Build the filing URL
                accession_clean = accession.replace("-", "")
                cik_clean = cik.lstrip("0") or "0"
                filing_url = f"https://www.sec.gov/Archives/edgar/{cik_clean}/{accession_clean}/{primary_doc}"
                
                results.append({
                    "ticker": ticker,
                    "form_type": form,
                    "filing_date": date,
                    "report_date": report_date,
                    "accession_number": accession,
                    "primary_document": primary_doc,
                    "filing_url": filing_url,
                    "sec_cgi_url": f"https://www.sec.gov/cgi-bin/viewer?action=view&cik={cik}&accession_number={accession}&xbrl_type=v",
                })
                
                logger.info(f"  {date}: {accession}")
                logger.info(f"    Filing URL: {filing_url}")
                
                count += 1
        
        logger.info(f"\n✅ Found {len(results)} {form_type} filings\n")
        return results
    
    except Exception as e:
        logger.error(f"❌ Error: {e}")
        return []

# Test it
submissions = get_filing_submissions("GOOGL", form_type="10-Q", limit=2)

2026-07-12 08:47:05,103 - INFO - 
2026-07-12 08:47:05,104 - INFO - ENDPOINT 3: Submissions for GOOGL
2026-07-12 08:47:05,104 - INFO - Form Type: 10-Q
2026-07-12 08:47:05,105 - INFO - ============================================================
2026-07-12 08:47:05,105 - INFO - URL: https://data.sec.gov/submissions/CIK0001652044.json

2026-07-12 08:47:05,333 - INFO -   2026-04-30: 0001652044-26-000048
2026-07-12 08:47:05,334 - INFO -     Filing URL: https://www.sec.gov/Archives/edgar/1652044/000165204426000048/goog-20260331.htm
2026-07-12 08:47:05,335 - INFO -   2025-10-30: 0001652044-25-000091
2026-07-12 08:47:05,335 - INFO -     Filing URL: https://www.sec.gov/Archives/edgar/1652044/000165204425000091/goog-20250930.htm
2026-07-12 08:47:05,336 - INFO - 
✅ Found 2 10-Q filings



---
# Complete Pipeline: Structured + Text Data

In [16]:
def build_complete_pipeline(ticker: str, quarters: int = 2):
    """
    Combine all three endpoints:
    
    1. Company Facts → Structured financial data (for queries)
    2. Company Concept → Specific metrics over time (for trends)
    3. Submissions → Filing links (for RAG with actual 10-Q text)
    """
    
    logger.info(f"\n\n" + "#"*70)
    logger.info(f"# COMPLETE PIPELINE: {ticker}")
    logger.info(f"#"*70)
    
    # Get structured data
    structured_data = get_all_financial_facts(ticker, limit_quarters=quarters)
    
    # Get revenue trend
    revenue_trend = get_concept_over_time(ticker, "Revenues")
    
    # Get filing links
    filing_links = get_filing_submissions(ticker, form_type="10-Q", limit=quarters)
    
    logger.info(f"\n{'='*60}")
    logger.info(f"PIPELINE COMPLETE FOR {ticker}")
    logger.info(f"{'='*60}")
    logger.info(f"\n✅ Structured Data (XBRL Facts): {len(structured_data)} quarters")
    logger.info(f"✅ Revenue Trend: {len(revenue_trend)} quarters")
    logger.info(f"✅ Filing Links (for RAG): {len(filing_links)} filings")
    
    return {
        "ticker": ticker,
        "structured_financials": structured_data,
        "revenue_trend": revenue_trend,
        "filing_links": filing_links,
    }

# Test complete pipeline
pipeline_result = build_complete_pipeline("MSFT", quarters=2)

2026-07-12 08:47:08,978 - INFO - 

######################################################################
2026-07-12 08:47:08,979 - INFO - # COMPLETE PIPELINE: MSFT
2026-07-12 08:47:08,980 - INFO - ######################################################################
2026-07-12 08:47:08,980 - INFO - 
2026-07-12 08:47:08,981 - INFO - ENDPOINT 1: Company Facts for MSFT
2026-07-12 08:47:08,982 - INFO - ============================================================
2026-07-12 08:47:08,982 - INFO - URL: https://data.sec.gov/api/xbrl/companyfacts/CIK0000789019.json

2026-07-12 08:47:09,354 - INFO - ✅ Found 1 quarters of structured financial data

2026-07-12 08:47:09,354 - INFO -   2010-12-31: Revenue $19,953,000,000
2026-07-12 08:47:09,358 - INFO - 
2026-07-12 08:47:09,359 - INFO - ENDPOINT 2: Company Concept for MSFT
2026-07-12 08:47:09,359 - INFO - Concept: Revenues
2026-07-12 08:47:09,359 - INFO - ============================================================
2026-07-12 08:47:09,359 - INFO -

---
# Summary: When to Use Each Endpoint

In [17]:
summary = """
╔════════════════════════════════════════════════════════════════════════════╗
║ SEC API: THREE ENDPOINTS QUICK REFERENCE                                    ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                             ║
║ ENDPOINT 1: Company Facts                                                  ║
║ https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json                   ║
║                                                                             ║
║ Use When:  You need ALL structured financial data (bulk ingestion)         ║
║ Response:  Complete XBRL data for company across all filings               ║
║ Size:      Large (~2-5MB) but comprehensive                                ║
║ Rate:      10 req/sec                                                      ║
║ Best For:  Data warehouse, complete financial history                      ║
║                                                                             ║
║─────────────────────────────────────────────────────────────────────────────║
║                                                                             ║
║ ENDPOINT 2: Company Concept                                                ║
║ https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{Concept}   ║
║                                                                             ║
║ Use When:  You need ONE metric over time (efficient, targeted)             ║
║ Response:  Just Revenue (or just Net Income, etc.) across quarters         ║
║ Size:      Small (~100KB)                                                  ║
║ Rate:      10 req/sec                                                      ║
║ Best For:  Specific metric trends, lightweight queries                     ║
║                                                                             ║
║─────────────────────────────────────────────────────────────────────────────║
║                                                                             ║
║ ENDPOINT 3: Submissions                                                    ║
║ https://data.sec.gov/submissions/CIK{cik}.json                             ║
║                                                                             ║
║ Use When:  You need filing links and metadata for RAG text extraction      ║
║ Response:  Filing index with dates, accessions, links to documents         ║
║ Size:      Small (~500KB)                                                  ║
║ Rate:      10 req/sec                                                      ║
║ Best For:  RAG pipeline (getting actual 10-Q/10-K text), metadata lookups  ║
║                                                                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║ YOUR PIPELINE:                                                              ║
║                                                                             ║
║ Phase 1 (Structured Data):                                                 ║
║   Use ENDPOINT 1 (Company Facts) → Get revenue, net income, margins        ║
║   Store in DuckDB with dlt                                                 ║
║                                                                             ║
║ Phase 2 (Text Data for RAG):                                               ║
║   Use ENDPOINT 3 (Submissions) → Get links to actual 10-Q documents        ║
║   Fetch the text and extract MD&A                                         ║
║   Store in DuckDB for RAG querying                                        ║
║                                                                             ║
║ Optional: Use ENDPOINT 2 (Company Concept) for specific metric trends      ║
║                                                                             ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(summary)


╔════════════════════════════════════════════════════════════════════════════╗
║ SEC API: THREE ENDPOINTS QUICK REFERENCE                                    ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                             ║
║ ENDPOINT 1: Company Facts                                                  ║
║ https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json                   ║
║                                                                             ║
║ Use When:  You need ALL structured financial data (bulk ingestion)         ║
║ Response:  Complete XBRL data for company across all filings               ║
║ Size:      Large (~2-5MB) but comprehensive                                ║
║ Rate:      10 req/sec                                                      ║
║ Best For:  Data warehouse, complete financial history                      ║
║                                               